# m07 lab · Inverse problems and regularization (Tikhonov, L-curve)

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

An electrode a height $h$ above the tissue records a smoothed copy of the tissue-level potential: across the
source-free blood layer each spatial harmonic of wavenumber $k$ is multiplied by $e^{-|k|h}$. Undoing that smoothing
is an inverse problem. You build the forward map, watch the naive inverse fail, then regularize it, choose the
regularization parameter three ways, and read Tikhonov regularization as a Bayesian estimate.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The forward map: continuation, the m03 front shape and the condition number | eq-m07-model, eq-m07-continuation |
| B | The naive inverse and the Picard plot | eq-m07-svd |
| C | Filters, the error budget and resolution | eq-m07-filters, eq-m07-error |
| D | Choosing lambda: discrepancy, L-curve, GCV | eq-m07-choice |
| E | The Bayesian reading of Tikhonov regularization | eq-m07-bayes |

Distances in mm, potentials in relative units. The tissue profile has three activation fronts of width 0.3 mm (as in
m03-d1), two of them 3 mm apart; heights and noise levels are free choices for illustration. Each part ends with
**checks**. No downloads; an Octave copy of parts A–E is in `labs/octave/m07_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import inverse as I
rng = np.random.default_rng(7)
n, L, w = 256, 32.0, 0.3                     # grid points, period (mm), front width (mm)
s, k = I.grid(n, L)
x = I.tissue_profile(s, w, L, n_harm=n // 2)  # the tissue-level truth

## Part A · The forward map

In a source-free layer the potential obeys Laplace's equation, and $e^{-kz}\cos kx$ is a solution: each harmonic
decays as $e^{-|k|h}$ between the tissue and an electrode at height $h$ (eq-m07-continuation). The m03 front shape
$s/(s^2 + a^2)$ has harmonics proportional to $e^{-|k|a}$, so continuing a front of width $w$ by $h$ gives exactly a
front of width $w + h$: the closed form of m03 is this smoothing. As a matrix, the map has condition number
$\kappa = e^{\pi h/\Delta x}$ (eq-m07-model): the gain at $k = 0$ over the gain at the shortest wavelength the grid
holds.

In [ ]:
dx = L / n
cont_err = max(np.max(np.abs(I.continue_up(x, k, h) - I.tissue_profile(s, w + h, L, n_harm=n // 2)))
               for h in (0.5, 1.0, 3.0))
print(f"continuing width {w} mm by h equals width {w} + h: largest difference {cont_err:.1e}")
kap = []
for h in (0.25, 0.5, 1.0, 2.0):
    sv = np.linalg.svd(I.continuation_matrix(n, L, h), compute_uv=False)
    kap.append((h, sv[0] / sv[-1], I.condition_number_continuation(h, dx)))
    print(f"h = {h} mm: condition number from the SVD {kap[-1][1]:.2e}, formula exp(pi h / dx) = {kap[-1][2]:.2e}")
for h in (1.0, 3.0):
    plt.plot(s, I.continue_up(x, k, h), label=f"recorded at h = {h} mm")
plt.plot(s, x, "k--", lw=1, label="tissue")
plt.xlabel("position (mm)"); plt.ylabel("potential (relative)"); plt.legend(); plt.show()

Beyond about $10^{16}$ the computed smallest singular values are rounding noise, so the SVD can no longer measure
the condition number: at $h$ = 2 mm the true value is near $10^{22}$.

In [ ]:
# Checks
assert cont_err < 1e-12
assert all(abs(a / b - 1) < 1e-5 for h, a, b in kap if h <= 0.5)
assert kap[-1][1] < 1e-3 * kap[-1][2]                    # h = 2 mm: the SVD saturates at rounding level

## Part B · The naive inverse and the Picard plot

Write $A = \sum_i s_i \mathbf u_i \mathbf v_i^\top$. The naive solution divides each data coefficient by its singular
value, so each noise coefficient is divided too (eq-m07-svd). For this operator the singular vectors are sines and
cosines and the singular values are the gains $e^{-|k|h}$. The **discrete Picard condition** asks that the data
coefficients fall faster than the singular values (Hansen 1990). Clean data meet it; noisy data level off at the noise
floor, about $\sigma\sqrt n$ per harmonic, and the ratio explodes.

In [ ]:
h = 1.0
A = I.continuation_matrix(n, L, h)
b0 = A @ x
sigma = 1e-3 * np.max(np.abs(b0))                        # 0.1% of the recorded peak
b = b0 + sigma * rng.standard_normal(n)
naive_err = np.linalg.norm(I.naive(A, b) - x) / np.linalg.norm(x)
print(f"naive inverse with 0.1% noise: relative error {naive_err:.1e}")
g = I.continuation_gain(k, h)
order = np.argsort(-g, kind="stable")
B0, B = np.abs(np.fft.fft(b0)), np.abs(np.fft.fft(b))
plt.semilogy(g[order], "k", label="singular values (gains)")
plt.semilogy(B0[order] / B0.max(), ".", ms=3, label="clean data coefficients")
plt.semilogy(B[order] / B0.max(), ".", ms=3, label="noisy data coefficients")
plt.axhline(sigma * np.sqrt(n) / B0.max(), color="gray", ls="--", label="noise level")
plt.ylim(1e-18, 10); plt.xlabel("index i (by decreasing singular value)"); plt.legend(fontsize=8); plt.show()
floor = np.median(B[g < 1e-6]) / (sigma * np.sqrt(n))
print(f"median noisy coefficient where the gain is below 1e-6: {floor:.2f} sigma sqrt(n)")

In [ ]:
# Checks
assert naive_err > 1e4
ok = g > 1e-6
assert np.allclose(B0[ok] / g[ok], np.abs(np.fft.fft(x))[ok], rtol=1e-6)   # clean ratio = tissue spectrum
assert 0.7 < floor < 1.0                                  # Rayleigh median: sqrt(ln 2) = 0.83

## Part C · Filters, the error budget and resolution

Regularization multiplies each term by a **filter factor**: 1 or 0 for truncated SVD, $s_i^2/(s_i^2 + \lambda^2)$ for
Tikhonov (eq-m07-filters). The expected squared error splits into **bias**, the detail removed, and **noise** passed
through (eq-m07-error). For this operator both are sums over harmonics. Check the formula by simulation, compare the
two filters at their best settings, and see what the best reconstruction can resolve as the electrode moves away.

In [ ]:
X = np.fft.fft(x)
xn = np.linalg.norm(x)
lams = np.logspace(-6, 0, 121)
h = 1.0
g = I.continuation_gain(k, h)
b0 = I.continue_up(x, k, h)
sigma = 1e-2 * np.max(np.abs(b0))                        # 1% of the recorded peak
parts = np.array([I.fourier_expected_error(g, X, sigma, I.tikhonov_factors(g, lam)) for lam in lams])
best = np.argmin(parts[:, 2]); lam_best = lams[best]
mc = np.mean([np.sum((I.fourier_tikhonov(b0 + sigma * rng.standard_normal(n), g, lam_best) - x) ** 2)
              for _ in range(300)])
order = np.argsort(np.abs(k), kind="stable")
tsvd = [I.fourier_expected_error(g, X, sigma, np.isin(np.arange(n), order[:m]).astype(float))[2] for m in range(1, n + 1)]
print(f"best lambda {lam_best:.1e}: expected relative error {np.sqrt(parts[best, 2]) / xn:.3f}, "
      f"simulated {np.sqrt(mc) / xn:.3f}; best truncated SVD {np.sqrt(min(tsvd)) / xn:.3f}")
for j, lab in enumerate(["bias", "noise", "total"]):
    plt.loglog(lams, np.sqrt(parts[:, j]) / xn, label=lab)
plt.axvline(lam_best, color="gray", ls="--"); plt.ylim(1e-3, 10)
plt.xlabel("lambda"); plt.ylabel("relative RMS error"); plt.legend(); plt.show()

**Resolution.** If the tissue pattern had the same amplitude in every harmonic and the noise in each harmonic were a
fraction $\varepsilon$ of it, a harmonic would be worth keeping only while $e^{-|k|h} > \varepsilon$, that is for
$|k| < \ln(1/\varepsilon)/h$. The shortest wavelength recovered, about $2\pi h/\ln(1/\varepsilon)$, grows in
proportion to the height. Below, the two fronts 3 mm apart are reconstructed at three heights with 1% noise.

In [ ]:
def n_peaks(y, lo=16.5, hi=22.5):
    seg = y[(s > lo) & (s < hi)]
    inner = seg[1:-1]
    return int(np.sum((inner > seg[:-2]) & (inner > seg[2:]) & (inner > 0.2 * seg.max())))

res = {}
fig, ax = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for a, h in zip(ax, (1.0, 2.0, 3.0)):
    g = I.continuation_gain(k, h)
    b0 = I.continue_up(x, k, h)
    sig = 1e-2 * np.max(np.abs(b0))
    tot = [I.fourier_expected_error(g, X, sig, I.tikhonov_factors(g, lam))[2] for lam in lams]
    xr = I.fourier_tikhonov(b0 + sig * rng.standard_normal(n), g, lams[np.argmin(tot)])
    res[h] = (n_peaks(b0), n_peaks(xr), 2 * np.pi * h / np.log(100))
    a.plot(s, x, "k--", lw=1); a.plot(s, b0, lw=1); a.plot(s, xr, lw=1.5)
    a.set_xlim(12, 26); a.set_title(f"h = {h} mm"); a.set_xlabel("position (mm)")
ax[0].set_ylabel("potential (relative)"); plt.show()
for h, (pm, pr, wl) in res.items():
    print(f"h = {h} mm: peaks between the close fronts recorded {pm}, reconstructed {pr}; rule-of-thumb shortest "
          f"wavelength {wl:.1f} mm")

In [ ]:
# Checks
assert abs(mc / parts[best, 2] - 1) < 0.1                # the error formula matches simulation
assert abs(np.sqrt(min(tsvd)) / np.sqrt(parts[best, 2]) - 1) < 0.15   # the two filters perform alike
assert res[1.0][0] == 1 and res[1.0][1] == 2             # at 1 mm: merged in the recording, separated after
assert res[3.0][1] == 1                                  # at 3 mm: lost

## Part D · Choosing lambda

In practice the truth is unknown. Three standard rules (eq-m07-choice): the **discrepancy principle** fits the data no
better than the noise allows and needs the noise level; the **L-curve** picks the corner of the curve of solution size
against residual (Hansen 1992; Hansen and O'Leary 1993); **generalized cross-validation** minimizes a predicted-error
function of the data (Golub, Heath and Wahba 1979). Here the truth is known, so you can score each rule by its error
relative to the best possible.

In [ ]:
lams = np.logspace(-7, 0, 141)
def scores(h, eps, seeds=range(5)):
    A = I.continuation_matrix(n, L, h)
    b0 = A @ x
    sig = eps * np.max(np.abs(b0))
    out = []
    for seed in seeds:
        b = b0 + sig * np.random.default_rng(seed).standard_normal(n)
        best = I.error_curve(A, b, x, lams).min()
        chosen = (I.discrepancy(A, b, sig * np.sqrt(n)), I.lcurve_corner(A, b, lams), lams[np.argmin(I.gcv(A, b, lams))])
        out.append([I.error_curve(A, b, x, [lam])[0] / best for lam in chosen])
    return np.array(out)

table = {eps: scores(1.0, eps) for eps in (1e-3, 1e-2, 1e-1)}
for eps, sc in table.items():
    print(f"h = 1 mm, noise {eps:.0e}: error / best error, median over 5 noise draws: "
          f"discrepancy {np.median(sc[:, 0]):.2f}, L-curve {np.median(sc[:, 1]):.2f}, GCV {np.median(sc[:, 2]):.2f}")
miss = scores(0.5, 1e-3)
print(f"h = 0.5 mm, noise 1e-03: L-curve error / best error, median {np.median(miss[:, 1]):.2f}")
A = I.continuation_matrix(n, L, 1.0)
b0 = A @ x; sig = 1e-2 * np.max(np.abs(b0)); b = b0 + sig * np.random.default_rng(0).standard_normal(n)
r, sn = I.residual_and_norm(A, b, lams)
plt.loglog(r, sn, "k", lw=1)
for lam, lab in ((I.discrepancy(A, b, sig * np.sqrt(n)), "discrepancy"), (I.lcurve_corner(A, b, lams), "L-curve corner"),
                 (lams[np.argmin(I.gcv(A, b, lams))], "GCV")):
    rr, ss = I.residual_and_norm(A, b, lam)
    plt.loglog(rr, ss, "o", label=lab)
plt.xlabel("residual norm ||A x - b||"); plt.ylabel("solution norm ||x||"); plt.legend(); plt.show()

In [ ]:
# Checks
assert all(np.median(sc, axis=0).max() < 1.35 for sc in table.values())
assert np.median(miss[:, 1]) > 2                          # the L-curve corner can miss badly

## Part E · The Bayesian reading

Suppose the unknown is drawn from a Gaussian prior $\mathbf x \sim \mathcal N(0, \sigma_x^2 I)$ and the noise is
Gaussian with standard deviation $\sigma_e$. Then the posterior mean is the Tikhonov solution with
$\lambda = \sigma_e/\sigma_x$, and the posterior covariance gives error bars (eq-m07-bayes). The error bars are only as
good as the prior: when it is right, 95% intervals contain the truth about 95% of the time.

In [ ]:
A = I.continuation_matrix(20, 10.0, 0.5)                # a small problem: 20 points over 10 mm, h = 0.5 mm
yy = A @ rng.standard_normal(20) + 0.01 * rng.standard_normal(20)
same = np.allclose(I.posterior(A, yy, 0.01, 1.0)[0], I.tikhonov(A, yy, 0.01 / 1.0))
same_cov = np.allclose(I.posterior(A, yy, 0.01, 1.0)[1], I.posterior_by_conditioning(A, yy, 0.01, 1.0)[1])
hits = []
for _ in range(600):
    xx = rng.standard_normal(20)
    y = A @ xx + 0.01 * rng.standard_normal(20)
    m, C = I.posterior(A, y, 0.01, 1.0)
    hits.append(np.abs(xx - m) <= 1.96 * np.sqrt(np.diag(C)))
coverage = np.mean(hits)
print(f"posterior mean equals Tikhonov with lambda = sigma_e / sigma_x: {same}; covariances agree: {same_cov}")
print(f"95% intervals contained the truth {100 * coverage:.1f}% of the time")
xx = rng.standard_normal(20); y = A @ xx + 0.01 * rng.standard_normal(20)
m, C = I.posterior(A, y, 0.01, 1.0); sd = np.sqrt(np.diag(C))
plt.fill_between(np.arange(20), m - 1.96 * sd, m + 1.96 * sd, alpha=0.3, label="95% interval")
plt.plot(m, label="posterior mean"); plt.plot(xx, "k.", label="truth")
plt.xlabel("grid point"); plt.ylabel("value"); plt.legend(); plt.show()

In [ ]:
# Checks
assert same and same_cov
assert 0.93 < coverage < 0.97

## Where to go next

- In part C, replace Tikhonov's identity penalty with a derivative penalty (multiply each harmonic's penalty by $k^2$).
  Which reconstruction keeps the fronts sharper?
- In part D, add noise that is correlated from sample to sample (smooth it first). Which rule suffers most? Hansen and
  O'Leary report that the L-curve is more robust than GCV to correlated errors.
- In part E, use a prior ten times too narrow. What happens to the coverage?
- m32 applies these tools to ECG imaging. Ridge regression in machine learning is the same estimate with a data
  matrix in place of $A$; Golub, Heath and Wahba introduced GCV for it.